# Tutorial: Ridge and Lasso Regression — Parameter Selection in scikit-learn
## DA5401W - Data Analytics Lab
**Instructor:** Dr. Arun B Ayyar

---

## About This Tutorial

This is  tutorial session on Ridge and Lasso Regression using scikit-learn. Each problem below provides:

1. A problem statement explaining what you need to do and why
2. Data creation code that is already written for you — just run it
3. A code cell with hints where you write your solution
4. A visualization cell that runs after your solution to show the results

The goal is to understand how different scikit-learn parameters affect Ridge and Lasso regression models.

---

## Setup — Run This First

Run the cell below to import all required libraries before starting the tutorial.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_diabetes, fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('All libraries loaded successfully!')

All libraries loaded successfully!


---
## Problem 1: Effect of `alpha` on Ridge Regression

### Problem Statement

The most important parameter in Ridge Regression is `alpha` (also called the regularization strength or penalty parameter). It controls how much the model penalizes large coefficients.

In scikit-learn, `Ridge(alpha=...)` accepts any positive float value:
- **Small alpha (e.g., 0.001):** Very little regularization — model behaves almost like OLS
- **Large alpha (e.g., 1000):** Very strong regularization — all coefficients are heavily shrunk toward zero

**What you need to do:**

Using the **Diabetes dataset** (a standard regression benchmark with 10 features), fit Ridge regression models for each alpha in the list `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`. For each model, compute the **test MSE** and the **L2 norm of the coefficients** (which measures how large the coefficients are). Store these in two lists.

**Key scikit-learn parameters for Ridge:**

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `alpha` | float | 1.0 | Regularization strength. Higher = more shrinkage |
| `fit_intercept` | bool | True | Whether to fit an intercept term |
| `solver` | str | 'auto' | Algorithm: 'auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga' |
| `max_iter` | int | None | Max iterations (for iterative solvers) |
| `tol` | float | 0.001 | Convergence tolerance |

**Expected outcome:** You should observe that as alpha increases, the coefficient norm decreases (shrinkage) and the test MSE first decreases then increases (U-shaped curve).

In [2]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# The Diabetes dataset has 10 features and 442 samples.
# We split 80% for training and 20% for testing.
# StandardScaler is applied so all features are on the same scale.

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit on train, transform train
X_test_s  = scaler.transform(X_test)       # only transform test (no fit)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Number of features: {X.shape[1]}')
print(f'Features: {feature_names}')

Training samples : 353
Test samples     : 89
Number of features: 10
Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


In [ ]:
# ── YOUR SOLUTION ───────────────────────────────────────────────────────────
# Fit Ridge regression for each alpha value and record test MSE and coef norm.

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

test_mse   = []   # store test MSE for each alpha
coef_norms = []   # store L2 norm of coefficients for each alpha

for alpha in alphas:
    # Hint: Create a Ridge model with the current alpha
    # ridge_model = Ridge(alpha=...)
    
    # Hint: Fit the model on the scaled training data
    # ridge_model.fit(...)
    
    # Hint: Predict on the scaled test data
    # y_pred = ridge_model.predict(...)
    
    # Hint: Compute test MSE using mean_squared_error(y_test, y_pred)
    # mse = ...
    
    # Hint: Compute L2 norm of coefficients using np.linalg.norm(ridge_model.coef_)
    # norm = ...
    
    # Hint: Append mse and norm to the respective lists
    pass

# Print a summary table
# Hint: Loop through alphas, test_mse, coef_norms and print each row
print(f"{'Alpha':>10}  {'Test MSE':>12}  {'Coef L2 Norm':>14}")
print('-' * 42)

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell visualizes the effect of alpha on test MSE and coefficient norm.
# It also compares Ridge against OLS (no regularization) as a baseline.

# OLS baseline
ols = LinearRegression().fit(X_train_s, y_train)
ols_mse  = mean_squared_error(y_test, ols.predict(X_test_s))
ols_norm = np.linalg.norm(ols.coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Alpha vs Test MSE
axes[0].semilogx(alphas, test_mse, 'o-', lw=2.5, ms=8, color='#E74C3C', label='Ridge')
axes[0].axhline(ols_mse, color='gray', ls='--', lw=2, label=f'OLS (no regularization)')
axes[0].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Test MSE', fontsize=11, fontweight='bold')
axes[0].set_title('Effect of Alpha on Test MSE', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].annotate('Optimal\nalpha', xy=(alphas[test_mse.index(min(test_mse))], min(test_mse)),
                 xytext=(alphas[test_mse.index(min(test_mse))]*5, min(test_mse)+200),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

# Right: Alpha vs Coefficient Norm
axes[1].semilogx(alphas, coef_norms, 'o-', lw=2.5, ms=8, color='#3498DB', label='Ridge')
axes[1].axhline(ols_norm, color='gray', ls='--', lw=2, label='OLS norm')
axes[1].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Coefficient L2 Norm', fontsize=11, fontweight='bold')
axes[1].set_title('Effect of Alpha on Coefficient Size', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Problem 1: Ridge Regression — Effect of Alpha\n'
             'Observation: As alpha increases, coefficients shrink and MSE forms a U-shape',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. Small alpha  → low shrinkage → coefficients close to OLS → may overfit')
print('  2. Optimal alpha → best test MSE → right balance of bias and variance')
print('  3. Large alpha  → heavy shrinkage → coefficients near zero → underfitting')
print('  4. Ridge NEVER sets any coefficient to exactly zero')

---
## Problem 2: Effect of `alpha` on Lasso Regression

### Problem Statement

Lasso Regression uses an L1 penalty instead of L2. The key difference is that Lasso can shrink coefficients to **exactly zero**, effectively performing automatic feature selection.

The `alpha` parameter in `Lasso(alpha=...)` works similarly to Ridge, but because Lasso uses coordinate descent (an iterative algorithm), you also need to set `max_iter` high enough to ensure convergence. The default `max_iter=1000` is often insufficient; use `max_iter=10000` as a safe default.

**What you need to do:**

Using the same Diabetes dataset, fit Lasso models for each alpha in `[0.001, 0.01, 0.1, 0.5, 1, 5, 10]`. For each model, record:
- **Test MSE**
- **Number of non-zero coefficients** (features that Lasso kept)
- **L1 norm** of the coefficients

**Key scikit-learn parameters for Lasso:**

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `alpha` | float | 1.0 | Regularization strength |
| `max_iter` | int | 1000 | Max iterations for coordinate descent — **increase to 10000** |
| `tol` | float | 0.0001 | Convergence tolerance |
| `selection` | str | 'cyclic' | Feature update order: 'cyclic' or 'random' |
| `warm_start` | bool | False | Reuse previous solution as starting point |
| `positive` | bool | False | Force all coefficients to be positive |

**Expected outcome:** As alpha increases, more and more coefficients become exactly zero. At very high alpha, all coefficients are zero (the model predicts the mean for every sample).

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Using the same Diabetes dataset from Problem 1.
# X_train_s, X_test_s, y_train, y_test are already created above.

lasso_alphas = [0.001, 0.01, 0.1, 0.5, 1, 5, 10]

print('Lasso alpha values to test:', lasso_alphas)
print(f'Total features in dataset: {X_train_s.shape[1]}')
print('Note: We expect Lasso to select fewer features as alpha increases.')

In [ ]:
# ── YOUR SOLUTION ───────────────────────────────────────────────────────────
# Fit Lasso regression for each alpha value.
# Record test MSE, number of non-zero coefficients, and L1 norm.

lasso_mse    = []   # store test MSE
nonzero_coef = []   # store count of non-zero coefficients
l1_norms     = []   # store L1 norm of coefficients

for alpha in lasso_alphas:
    # Hint: Create a Lasso model with the current alpha and max_iter=10000
    # lasso_model = Lasso(alpha=..., max_iter=10000)
    
    # Hint: Fit on scaled training data
    # lasso_model.fit(...)
    
    # Hint: Predict on scaled test data
    # y_pred = lasso_model.predict(...)
    
    # Hint: Compute test MSE
    # mse = mean_squared_error(...)
    
    # Hint: Count non-zero coefficients using np.sum(lasso_model.coef_ != 0)
    # nz = ...
    
    # Hint: Compute L1 norm using np.sum(np.abs(lasso_model.coef_))
    # l1 = ...
    
    pass

# Print summary table
print(f"{'Alpha':>8}  {'Test MSE':>12}  {'Non-zero Coefs':>16}  {'L1 Norm':>10}")
print('-' * 54)

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell shows two key effects of alpha in Lasso:
# (1) How test MSE changes with alpha
# (2) How many features Lasso keeps (non-zero coefficients) as alpha increases

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Alpha vs Test MSE
axes[0].semilogx(lasso_alphas, lasso_mse, 'o-', lw=2.5, ms=8, color='#9B59B6', label='Lasso')
axes[0].axhline(ols_mse, color='gray', ls='--', lw=2, label='OLS baseline')
axes[0].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Test MSE', fontsize=11, fontweight='bold')
axes[0].set_title('Lasso: Alpha vs Test MSE', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: Alpha vs Number of Non-zero Coefficients
axes[1].semilogx(lasso_alphas, nonzero_coef, 's-', lw=2.5, ms=8, color='#E67E22')
axes[1].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Number of Non-zero Coefficients', fontsize=11, fontweight='bold')
axes[1].set_title('Lasso: Feature Selection vs Alpha', fontsize=12, fontweight='bold')
axes[1].set_ylim(-0.5, X_train_s.shape[1] + 0.5)
axes[1].axhline(X_train_s.shape[1], color='gray', ls='--', lw=1.5, label='All features (OLS)')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Problem 2: Lasso Regression — Effect of Alpha\n'
             'Key difference from Ridge: Lasso sets coefficients to EXACTLY zero',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. As alpha increases, Lasso removes features one by one')
print('  2. At high alpha, all coefficients become exactly zero (null model)')
print('  3. This is DIFFERENT from Ridge which only shrinks but never zeros out')
print('  4. Lasso performs automatic feature selection — a major advantage')

---
## Problem 3: Choosing Optimal Alpha Using Cross-Validation

### Problem Statement

In Problems 1 and 2, you manually tested a few alpha values. In practice, you should **never choose alpha by looking at test set performance** — that would cause data leakage and give an overly optimistic estimate.

The correct approach is **k-fold cross-validation on the training set**. scikit-learn provides two convenient classes for this:

- `RidgeCV(alphas=[...], cv=k)` — fits Ridge for each alpha and returns the best one
- `LassoCV(cv=k, max_iter=...)` — fits Lasso along a regularization path and returns the best alpha

**What you need to do:**

Using the Diabetes dataset:
1. Use `RidgeCV` with `alphas=np.logspace(-3, 4, 100)` and `cv=5` to find the best Ridge alpha
2. Use `LassoCV` with `cv=5` and `max_iter=10000` to find the best Lasso alpha
3. For both models, report: optimal alpha, test MSE, R² score, and (for Lasso) selected features

**Key parameters:**

| Class | Key Parameter | Description |
|-------|--------------|-------------|
| `RidgeCV` | `alphas` | Array of alpha values to search over |
| `RidgeCV` | `cv` | Number of cross-validation folds |
| `LassoCV` | `cv` | Number of cross-validation folds |
| `LassoCV` | `max_iter` | Must be large enough for convergence |
| Both | `.alpha_` | **Attribute** — the best alpha found by CV |

**Expected outcome:** The optimal alpha found by CV should give better or equal test MSE compared to the manually selected alphas in Problems 1 and 2.

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Using the same Diabetes dataset.
# Define the search grid for alpha values.

alpha_grid = np.logspace(-3, 4, 100)  # 100 values from 0.001 to 10000
print(f'Alpha search range: {alpha_grid[0]:.4f} to {alpha_grid[-1]:.0f}')
print(f'Total alpha values to search: {len(alpha_grid)}')
print(f'Cross-validation folds: 5')

In [ ]:
# ── YOUR SOLUTION ───────────────────────────────────────────────────────────
# Use RidgeCV and LassoCV to automatically find the best alpha.

# --- Part A: RidgeCV ---
# Hint: ridge_cv = RidgeCV(alphas=alpha_grid, cv=5)
# Hint: ridge_cv.fit(X_train_s, y_train)
# Hint: Access best alpha with ridge_cv.alpha_

# ridge_cv = ...
# ridge_cv.fit(...)

# ridge_test_mse = mean_squared_error(y_test, ridge_cv.predict(X_test_s))
# ridge_r2       = r2_score(y_test, ridge_cv.predict(X_test_s))

# print(f'RidgeCV optimal alpha : {ridge_cv.alpha_:.4f}')
# print(f'RidgeCV test MSE      : {ridge_test_mse:.4f}')
# print(f'RidgeCV R2 score      : {ridge_r2:.4f}')


# --- Part B: LassoCV ---
# Hint: lasso_cv = LassoCV(cv=5, max_iter=10000, random_state=42)
# Hint: lasso_cv.fit(X_train_s, y_train)
# Hint: Access best alpha with lasso_cv.alpha_
# Hint: Count selected features with np.sum(lasso_cv.coef_ != 0)

# lasso_cv = ...
# lasso_cv.fit(...)

# lasso_test_mse = mean_squared_error(y_test, lasso_cv.predict(X_test_s))
# lasso_r2       = r2_score(y_test, lasso_cv.predict(X_test_s))
# selected       = [feature_names[i] for i in np.where(lasso_cv.coef_ != 0)[0]]

# print(f'LassoCV optimal alpha  : {lasso_cv.alpha_:.4f}')
# print(f'LassoCV test MSE       : {lasso_test_mse:.4f}')
# print(f'LassoCV R2 score       : {lasso_r2:.4f}')
# print(f'LassoCV selected features: {selected}')

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell shows the cross-validation path for both Ridge and Lasso,
# highlighting the optimal alpha chosen by each method.

# Compute CV scores manually for Ridge (for plotting)
ridge_cv_mse = [-cross_val_score(Ridge(alpha=a), X_train_s, y_train, cv=5,
                 scoring='neg_mean_squared_error').mean() for a in alpha_grid]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: RidgeCV path
axes[0].semilogx(alpha_grid, ridge_cv_mse, lw=2, color='#E74C3C')
axes[0].axvline(ridge_cv.alpha_, color='red', ls='--', lw=2,
                label=f'Optimal alpha = {ridge_cv.alpha_:.4f}')
axes[0].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('CV Mean Squared Error', fontsize=11, fontweight='bold')
axes[0].set_title('RidgeCV: Cross-Validation Path', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: LassoCV path
axes[1].semilogx(lasso_cv.alphas_, lasso_cv.mse_path_.mean(axis=1), lw=2, color='#9B59B6')
axes[1].axvline(lasso_cv.alpha_, color='purple', ls='--', lw=2,
                label=f'Optimal alpha = {lasso_cv.alpha_:.4f}')
axes[1].set_xlabel('Alpha (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('CV Mean Squared Error', fontsize=11, fontweight='bold')
axes[1].set_title('LassoCV: Cross-Validation Path', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Problem 3: Optimal Alpha via 5-Fold Cross-Validation\n'
             'The dashed line marks the alpha that minimizes CV error',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. The CV path has a clear minimum — this is the optimal alpha')
print('  2. Left of minimum: underfitting (too much regularization)')
print('  3. Right of minimum: overfitting (too little regularization)')
print('  4. Always use CV to select alpha — never use the test set')

---
## Problem 4: Ridge vs Lasso on Multicollinear Data

### Problem Statement

Multicollinearity occurs when two or more features are highly correlated with each other. This causes OLS regression to produce **unstable, high-variance coefficients** — small changes in the data can lead to very different coefficient estimates.

**Ridge regression is specifically designed to handle multicollinearity.** By adding an L2 penalty, Ridge distributes the coefficient weight among correlated features rather than assigning arbitrary large values to them.

**Lasso behaves differently:** it tends to arbitrarily pick one feature from a correlated group and set the others to zero.

**What you need to do:**

A synthetic dataset has been created with 6 features where features 4, 5, and 6 are near-duplicates of features 1, 2, and a combination of 1+2. The true coefficients are `[2.0, -1.5, 0.8, 0.0, 0.0, 0.0]` (only the first 3 features matter).

1. Fit OLS, Ridge (using `RidgeCV`), and Lasso (using `LassoCV`) on this dataset
2. Compare the estimated coefficients against the true coefficients
3. Compare test MSE for all three models

**Expected outcome:** OLS will have unstable, inaccurate coefficients. Ridge will produce stable estimates. Lasso may select only one of each correlated pair.

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Synthetic multicollinear dataset:
# - 6 features, 200 samples
# - Features 4, 5, 6 are near-copies of features 1, 2, and (1+2)
# - True coefficients: [2.0, -1.5, 0.8, 0.0, 0.0, 0.0]

np.random.seed(42)
n = 200
X_base = np.random.randn(n, 3)
X_mc = np.column_stack([
    X_base[:, 0],                                           # Feature 1
    X_base[:, 1],                                           # Feature 2
    X_base[:, 2],                                           # Feature 3
    X_base[:, 0] + 0.05 * np.random.randn(n),              # Feature 4 ~ Feature 1
    X_base[:, 1] + 0.05 * np.random.randn(n),              # Feature 5 ~ Feature 2
    X_base[:, 0] + X_base[:, 1] + 0.1 * np.random.randn(n) # Feature 6 ~ Feature 1+2
])
true_coefs = np.array([2.0, -1.5, 0.8, 0.0, 0.0, 0.0])
y_mc = X_mc @ true_coefs + 0.5 * np.random.randn(n)

X_mc_tr, X_mc_te, y_mc_tr, y_mc_te = train_test_split(X_mc, y_mc, test_size=0.2, random_state=42)
sc_mc = StandardScaler()
X_mc_tr_s = sc_mc.fit_transform(X_mc_tr)
X_mc_te_s = sc_mc.transform(X_mc_te)

print('Dataset created successfully!')
print(f'Shape: {X_mc.shape}')
print(f'True coefficients: {true_coefs}')

# Show correlation matrix
corr = pd.DataFrame(X_mc, columns=[f'X{i+1}' for i in range(6)]).corr()
print('\nCorrelation between features:')
print(corr.round(2).to_string())

In [ ]:
# ── YOUR SOLUTION ───────────────────────────────────────────────────────────
# Fit OLS, Ridge (with RidgeCV), and Lasso (with LassoCV) on the multicollinear dataset.

# --- OLS ---
# Hint: ols_mc = LinearRegression().fit(X_mc_tr_s, y_mc_tr)
# ols_mc = ...

# --- Ridge with cross-validation ---
# Hint: ridge_mc = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(X_mc_tr_s, y_mc_tr)
# ridge_mc = ...

# --- Lasso with cross-validation ---
# Hint: lasso_mc = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_mc_tr_s, y_mc_tr)
# lasso_mc = ...

# --- Print comparison ---
# Hint: Print true_coefs, ols_mc.coef_, ridge_mc.coef_, lasso_mc.coef_
# Hint: Print test MSE for each model
# print(f'True coefficients : {true_coefs}')
# print(f'OLS coefficients  : {np.round(ols_mc.coef_, 3)}')
# print(f'Ridge coefficients: {np.round(ridge_mc.coef_, 3)}')
# print(f'Lasso coefficients: {np.round(lasso_mc.coef_, 3)}')

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell shows:
# (1) Coefficient comparison: True vs OLS vs Ridge vs Lasso
# (2) Correlation heatmap of the multicollinear features

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Coefficient comparison
x = np.arange(6)
w = 0.18
axes[0].bar(x - 1.5*w, true_coefs,       w, label='True',  color='black',   alpha=0.9)
axes[0].bar(x - 0.5*w, ols_mc.coef_,    w, label='OLS',   color='#E74C3C', alpha=0.8)
axes[0].bar(x + 0.5*w, ridge_mc.coef_,  w, label='Ridge', color='#3498DB', alpha=0.8)
axes[0].bar(x + 1.5*w, lasso_mc.coef_,  w, label='Lasso', color='#2ECC71', alpha=0.8)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'X{i+1}' for i in range(6)])
axes[0].set_xlabel('Feature', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Coefficient Value', fontsize=11, fontweight='bold')
axes[0].set_title('Coefficients: True vs Estimated\n(OLS is unstable; Ridge is stable)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Right: Correlation heatmap
corr_df = pd.DataFrame(X_mc, columns=[f'X{i+1}' for i in range(6)])
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], square=True)
axes[1].set_title('Feature Correlation Matrix\n(X4~X1, X5~X2, X6~X1+X2)', fontsize=11, fontweight='bold')

plt.suptitle('Problem 4: Multicollinearity — Why Ridge is Better\n'
             'Ridge stabilizes coefficients; OLS produces large, unstable values',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. OLS coefficients are far from the true values (unstable due to multicollinearity)')
print('  2. Ridge distributes weight among correlated features — closer to true values')
print('  3. Lasso picks one from each correlated group and zeros out the others')
print('  4. Ridge is the preferred model when features are highly correlated')

---
## Problem 5: Ridge vs Lasso on Sparse Data (Feature Selection)

### Problem Statement

In many real-world problems, you have many features but only a few are truly relevant. This is called a **sparse** setting. For example, in genomics you might have thousands of genes but only a handful are related to a disease.

**Lasso excels in sparse settings** because it automatically sets irrelevant feature coefficients to exactly zero. Ridge, on the other hand, keeps all features but with small coefficients — it does not perform feature selection.

**What you need to do:**

A synthetic dataset has been created with **50 features** but only **5 are informative** (features 0–4 have non-zero true coefficients; features 5–49 are pure noise).

1. Fit OLS, Ridge (`RidgeCV`), and Lasso (`LassoCV`) on this dataset
2. Report test MSE for all three models
3. Report which features Lasso selects (non-zero coefficients)
4. Check whether Lasso correctly identifies the 5 informative features

**Expected outcome:** Lasso should select approximately the 5 informative features and achieve better test MSE than OLS. Ridge will keep all 50 features.

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Synthetic sparse dataset:
# - 50 features, 300 samples
# - Only features 0-4 have non-zero true coefficients
# - Features 5-49 are pure noise

np.random.seed(42)
n_samples, n_features, n_informative = 300, 50, 5

X_sp = np.random.randn(n_samples, n_features)
true_coefs_sp = np.zeros(n_features)
true_coefs_sp[:n_informative] = [3.0, -2.5, 1.8, -1.2, 2.2]  # only first 5 are non-zero

y_sp = X_sp @ true_coefs_sp + 0.5 * np.random.randn(n_samples)

X_sp_tr, X_sp_te, y_sp_tr, y_sp_te = train_test_split(X_sp, y_sp, test_size=0.2, random_state=42)
sc_sp = StandardScaler()
X_sp_tr_s = sc_sp.fit_transform(X_sp_tr)
X_sp_te_s = sc_sp.transform(X_sp_te)

print('Sparse dataset created!')
print(f'Total features: {n_features}')
print(f'Informative features (indices 0-4): {n_informative}')
print(f'True non-zero coefficients: {true_coefs_sp[:5]}')
print(f'Noise features (indices 5-49): {n_features - n_informative}')

In [ ]:
# ── YOUR SOLUTION ───────────────────────────────────────────────────────────
# Fit OLS, Ridge, and Lasso on the sparse dataset.

# --- OLS ---
# Hint: ols_sp = LinearRegression().fit(X_sp_tr_s, y_sp_tr)
# ols_sp = ...

# --- Ridge with cross-validation ---
# Hint: ridge_sp = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(X_sp_tr_s, y_sp_tr)
# ridge_sp = ...

# --- Lasso with cross-validation ---
# Hint: lasso_sp = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_sp_tr_s, y_sp_tr)
# lasso_sp = ...

# --- Print results ---
# Hint: Print test MSE for each model
# Hint: Use np.where(lasso_sp.coef_ != 0)[0] to find selected feature indices
# Hint: Check if set(range(5)).issubset(set(selected_indices)) to verify correct selection

# print(f'Test MSE - OLS  : ...')
# print(f'Test MSE - Ridge: ...')
# print(f'Test MSE - Lasso: ...')
# print(f'Lasso selected features: ...')

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell shows:
# (1) Coefficient comparison across all 50 features — Lasso zeroes out noise features
# (2) True vs estimated coefficients for the first 10 features

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: All 50 coefficients
x = np.arange(n_features)
axes[0].bar(x, np.abs(ols_sp.coef_),    alpha=0.5, label='OLS (absolute)', color='#E74C3C')
axes[0].bar(x, np.abs(lasso_sp.coef_),  alpha=0.85, label='Lasso (absolute)', color='#3498DB')
axes[0].axvline(4.5, color='green', ls='--', lw=2, label='Informative features (0-4)')
axes[0].set_xlabel('Feature Index', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Absolute Coefficient Value', fontsize=11, fontweight='bold')
axes[0].set_title('Lasso Zeros Out Noise Features\n(OLS spreads weight across all 50 features)',
                  fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Right: True vs estimated for first 10 features
x10 = np.arange(10)
w = 0.2
axes[1].bar(x10 - w,   true_coefs_sp[:10],   w, label='True',  color='black',   alpha=0.9)
axes[1].bar(x10,       ols_sp.coef_[:10],    w, label='OLS',   color='#E74C3C', alpha=0.8)
axes[1].bar(x10 + w,   lasso_sp.coef_[:10],  w, label='Lasso', color='#3498DB', alpha=0.8)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xticks(x10)
axes[1].set_xticklabels([f'X{i}' for i in range(10)])
axes[1].set_xlabel('Feature', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Coefficient Value', fontsize=11, fontweight='bold')
axes[1].set_title('True vs Estimated (First 10 Features)\n(X5-X9 should be zero)',
                  fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Problem 5: Sparse Data — Lasso Feature Selection\n'
             'Lasso correctly identifies the 5 informative features and zeros out the rest',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. OLS assigns non-zero coefficients to ALL 50 features (overfitting noise)')
print('  2. Lasso correctly zeros out most noise features')
print('  3. Lasso achieves better test MSE by ignoring irrelevant features')
print('  4. Use Lasso when you expect many features to be irrelevant')

---
## Problem 6: Full Pipeline on California Housing Dataset

### Problem Statement

In this final problem, you will apply everything you have learned to a real-world dataset: the **California Housing dataset** (20,640 samples, 8 features). The task is to predict median house values based on features like median income, house age, and location.

**What you need to do:**

Build a complete regression pipeline:
1. Load and split the data (80/20 train-test split)
2. Scale the features using `StandardScaler`
3. Fit OLS, `RidgeCV`, and `LassoCV` models
4. Report test MSE, MAE, and R² for all three models in a comparison table
5. Identify which features Lasso selects and which it drops

**Key scikit-learn parameters to use:**

| Step | Class/Function | Key Parameters |
|------|---------------|----------------|
| Split | `train_test_split` | `test_size=0.2`, `random_state=42` |
| Scale | `StandardScaler` | `fit_transform` on train, `transform` on test |
| Ridge | `RidgeCV` | `alphas=np.logspace(-3, 4, 50)`, `cv=5` |
| Lasso | `LassoCV` | `cv=5`, `max_iter=10000` |
| Metrics | `mean_squared_error`, `mean_absolute_error`, `r2_score` | — |

**Expected outcome:** All three models should achieve R² > 0.6. Ridge and Lasso should perform similarly to OLS since the 8 features are well-designed.

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# California Housing dataset from scikit-learn.
# Features: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
# Target: Median house value (in $100,000s)

from sklearn.metrics import mean_absolute_error

housing = fetch_california_housing()
X_cal, y_cal = housing.data, housing.target
cal_features = housing.feature_names

print('California Housing Dataset')
print(f'  Samples  : {X_cal.shape[0]}')
print(f'  Features : {X_cal.shape[1]}')
print(f'  Feature names: {list(cal_features)}')
print(f'  Target range: {y_cal.min():.2f} to {y_cal.max():.2f} ($100k)')

In [ ]:
# ── YOUR SOLUTION ───────────────────────────────────────────────────────────
# Build the full pipeline: split → scale → fit → evaluate

# Step 1: Split the data
# Hint: X_cal_tr, X_cal_te, y_cal_tr, y_cal_te = train_test_split(X_cal, y_cal, test_size=0.2, random_state=42)
# X_cal_tr, X_cal_te, y_cal_tr, y_cal_te = ...

# Step 2: Scale the features
# Hint: sc_cal = StandardScaler()
# Hint: X_cal_tr_s = sc_cal.fit_transform(X_cal_tr)
# Hint: X_cal_te_s  = sc_cal.transform(X_cal_te)
# sc_cal = ...

# Step 3: Fit OLS, RidgeCV, LassoCV
# ols_cal   = LinearRegression().fit(...)
# ridge_cal = RidgeCV(alphas=np.logspace(-3, 4, 50), cv=5).fit(...)
# lasso_cal = LassoCV(cv=5, max_iter=10000, random_state=42).fit(...)

# Step 4: Build a comparison DataFrame
# Hint: Use pd.DataFrame with columns: Model, Alpha, Train MSE, Test MSE, MAE, R2
# results = pd.DataFrame({...})
# print(results.to_string(index=False))

# Step 5: Report Lasso feature selection
# Hint: selected = [cal_features[i] for i in np.where(lasso_cal.coef_ != 0)[0]]
# dropped  = [cal_features[i] for i in np.where(lasso_cal.coef_ == 0)[0]]
# print(f'Lasso selected : {selected}')
# print(f'Lasso dropped  : {dropped}')

In [ ]:
# ── VISUALIZATION (Run after your solution above) ───────────────────────────
# This cell shows:
# (1) Feature importance (absolute coefficients) for all three models
# (2) Predicted vs actual values for Ridge and Lasso on the test set

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Feature importance
x = np.arange(len(cal_features))
w = 0.25
axes[0].bar(x - w,  np.abs(ols_cal.coef_),   w, label='OLS',   alpha=0.85, color='#E74C3C')
axes[0].bar(x,      np.abs(ridge_cal.coef_),  w, label='Ridge', alpha=0.85, color='#3498DB')
axes[0].bar(x + w,  np.abs(lasso_cal.coef_),  w, label='Lasso', alpha=0.85, color='#2ECC71')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f[:8] for f in cal_features], rotation=30, ha='right')
axes[0].set_xlabel('Feature', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Absolute Coefficient', fontsize=11, fontweight='bold')
axes[0].set_title('Feature Importance: OLS vs Ridge vs Lasso', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Right: Predicted vs Actual
y_pred_ridge = ridge_cal.predict(X_cal_te_s)
y_pred_lasso = lasso_cal.predict(X_cal_te_s)
axes[1].scatter(y_cal_te, y_pred_ridge, alpha=0.2, s=8, color='#3498DB', label='Ridge')
axes[1].scatter(y_cal_te, y_pred_lasso, alpha=0.2, s=8, color='#2ECC71', label='Lasso')
lims = [min(y_cal_te.min(), y_pred_ridge.min()), max(y_cal_te.max(), y_pred_ridge.max())]
axes[1].plot(lims, lims, 'k--', lw=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Value ($100k)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Predicted Value ($100k)', fontsize=11, fontweight='bold')
axes[1].set_title('Predicted vs Actual (Test Set)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Problem 6: California Housing — Full Pipeline Results\n'
             'MedInc (Median Income) is the most important predictor',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('What to observe:')
print('  1. MedInc (Median Income) has the largest coefficient in all models')
print('  2. Ridge and Lasso perform similarly to OLS on this well-designed dataset')
print('  3. Regularization is most beneficial when data is noisy or high-dimensional')
print('  4. The predictions cluster around the diagonal — indicating good fit')

---
## Summary: Key scikit-learn Parameters for Ridge and Lasso

### Ridge Regression (`sklearn.linear_model.Ridge`)

| Parameter | Default | What It Controls | When to Change |
|-----------|---------|-----------------|----------------|
| `alpha` | 1.0 | Regularization strength | Always tune via `RidgeCV` |
| `fit_intercept` | True | Whether to fit a bias term | Set False only if data is centered |
| `solver` | 'auto' | Optimization algorithm | Use 'sag'/'saga' for large datasets |
| `max_iter` | None | Max iterations (iterative solvers) | Increase if solver doesn't converge |
| `tol` | 0.001 | Convergence tolerance | Decrease for more precise solution |

### Lasso Regression (`sklearn.linear_model.Lasso`)

| Parameter | Default | What It Controls | When to Change |
|-----------|---------|-----------------|----------------|
| `alpha` | 1.0 | Regularization strength | Always tune via `LassoCV` |
| `fit_intercept` | True | Whether to fit a bias term | Set False only if data is centered |
| `max_iter` | 1000 | Max coordinate descent iterations | **Always set to 10000** |
| `tol` | 0.0001 | Convergence tolerance | Decrease for more precise solution |
| `selection` | 'cyclic' | Feature update order | Use 'random' for faster convergence |
| `warm_start` | False | Reuse previous solution | Set True when fitting many alphas |
| `positive` | False | Force positive coefficients | Set True if domain requires it |

### When to Use Ridge vs Lasso

| Scenario | Use Ridge | Use Lasso |
|----------|-----------|----------|
| Features are highly correlated | ✓ | |
| Many irrelevant features | | ✓ |
| Need feature selection / interpretability | | ✓ |
| All features are expected to matter | ✓ | |
| High-dimensional data (p >> n) | | ✓ |
| Stable, smooth coefficient estimates | ✓ | |

---
**Course:** DA5401W - Data Analytics Lab  |  **Instructor:** Dr. Arun B Ayyar